# Tic toc toe

In [1]:
import numpy as np

"""

"""

# Game board
board = np.zeros(9, dtype=np.int8)

def display_board(board):
    # display_board = board.reshape(3,3)
    # for loop...

    # Simple solution
    print(f"{board[0]} | {board[1]} | {board[2]}")
    print(f"{board[3]} | {board[4]} | {board[5]}")
    print(f"{board[6]} | {board[7]} | {board[8]}")
    print()

def display_board_characters(board):
    # strBoard = [str(i) for i in board]
    strBoard = np.empty(9, dtype='U1')
    strBoard[np.where(board == 1)[0]] = 'X'
    strBoard[np.where(board == 0)[0]] = ' '
    strBoard[np.where(board == -1)[0]] = 'O'

    # Simple solution
    display_board(board=strBoard)


board[0] = 1
board[1] = 1
board[2] = 1

board[8] = -1

print(f"Game board {board}")

display_board(board=board)
display_board_characters(board=board)

Game board [ 1  1  1  0  0  0  0  0 -1]
1 | 1 | 1
0 | 0 | 0
0 | 0 | -1

X | X | X
  |   |  
  |   | O



In [2]:
def encode_array(arrIndices):
    # Function encodes array of indices into array of ones
    #
    #   arrIndices: Indices array
    #
    rtn = np.zeros(9)
    rtn[arrIndices] = 1
    return rtn

def check_game_complete(board, solutionMatrix):

    depth = np.sum(np.abs(board))   # No. of turns taken

    # Look at results
    scores = solutionMatrix.dot(board)

    if np.max(scores) == 3:     # X has won
        return 10 - depth
    elif np.min(scores) == -3:  # O has won
        return -10 + depth
    elif depth == 9:            # Draw
        return 0

    return None     # Game not finished

# Define solutions as indices of the board    
solutions = [[0, 1, 2], [3, 4, 5], [6, 7, 8],   # Rows
             [0, 3, 6], [1, 4, 7], [2, 5, 8],   # Columns
             [0, 4, 8], [2, 4, 6]]              # Diagonals

# Encode indices into arrays of 1s & 0s to form solutionMatrix
solMat = np.apply_along_axis(encode_array, axis=1, arr=solutions, )

testBoardXWin = [1, 1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardXWin, solutionMatrix=solMat))

testBoardNone = [1, -1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardNone, solutionMatrix=solMat))

testBoardOWin = [-1, 1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardOWin, solutionMatrix=solMat))

testBoardDraw = [1, -1, 1, -1, -1, 1, -1, 1, -1]
print(check_game_complete(board=testBoardDraw, solutionMatrix=solMat))

4
None
-4
0


In [3]:
def random_player(board, XorO):
    """
    Player function - Takes in the current board and return a move.

    random_player: Move is randomly selected

    Args:
        board (array(9, int)): Array of int that represent the current board
        XorO (int): value of 1 or -1, corresponding to X and O respectively

    Returns:
        int: index of move with respect to board
    """
    possibleMoves = np.where(board == 0)[0]
    move = np.random.choice(possibleMoves)
    return move

In [4]:
# Game loop

def playGame(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX(plyBoard, player)] = player
        else: # player == -1
            plyBoard[playerO(plyBoard, player)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print("Game is won by X (1)")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print("Game is won by O (-1)")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print("Game is a draw")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return None # Will be updated when I need game data for Neural Network

In [5]:
# Random vs Random players

noGames = 5
# Loop number of games
for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGame(playerX=random_player, playerO=random_player, solMat=solMat, startBoard=rndBoard)

Game is won by X (1)
X | O | X
X | O | O
X | X | O

Game is won by X (1)
  | X | O
  | X |  
O | X |  

Game is won by X (1)
O | O | X
O | X | O
X | X | X

Game is won by X (1)
X |   | O
X | O | O
X | X |  

Game is won by X (1)
X |   | X
  | X | O
X | O | O



## Move to class structure

In [6]:
# Abstract class
from abc import ABC, abstractmethod

# Blueprint for tic tac toe player
class player(ABC):
    @abstractmethod
    def nextMove(self, board):
        """
        Calculated next move based on current state (board)
        Args:
            board (array(9, int)): Array of int that represent the current board
        
        Returns:
            int: index of move with respect to board
            
        """
        pass

In [7]:
# Random player
class RandomPlayer(player):
    
    def __init__(self):
        pass

    def nextMove(self, board):
        possibleMoves = np.where(board == 0)[0]
        move = np.random.choice(possibleMoves)
        return move

In [127]:
def playGameClass(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX.nextMove(plyBoard)] = player
        else: # player == -1
            plyBoard[playerO.nextMove(plyBoard)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print(f"Game is won by X (1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print(f"Game is won by O (-1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print(f"Game is a draw, first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return None # Will be updated when I need game data for Neural Network

In [9]:
player1X = RandomPlayer()
player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard)

Game is a draw
X | O | O
O | X | X
X | X | O

Game is won by X (1)
O |   | X
O | O | X
X |   | X

Game is a draw
O | X | O
O | X | X
X | O | X

Game is a draw
O | X | X
X | X | O
O | O | X

Game is a draw
X | O | X
O | O | X
X | X | O



In [ ]:
# MaxMin player
class MinMaxPlayer(player):
    
    def __init__(self, playerType, solutionMatrix):
        self.playerType = playerType
        self.solutionMatrix = solutionMatrix
    
    def _minmax(self, board, currentPlayer):
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=board, solutionMatrix=self.solutionMatrix)
    
        if result is None:
            
            # Game not concluded, recursively try next move with _minmax
            bestScore = -np.inf #* currentPlayer 
            possMoves = np.where(board == 0)[0]

            for move in possMoves:

                testBoard = board.copy()
                # Try a move
                testBoard[move] = currentPlayer

                # Recursive call of _minmax
                # Note: 'currentPlayer *' ensure bestScore is always a positive value
                bestScore = max(bestScore, 
                                (currentPlayer * self._minmax(board = testBoard, 
                                                              currentPlayer = -currentPlayer)))

            return bestScore * currentPlayer    
        elif result > 0:
            return result             
        elif result < 0:
            return result
        elif result == 0:
            return result
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)

    def nextMove(self, board):
        
        possibleMoves = np.where(board == 0)[0]
        bestValue = -np.inf
        bestMove = possibleMoves[0]
        scoreTracker = np.ones(9) - 100

        for move in possibleMoves:
            testBoard = board.copy()
            
            # Try a move
            testBoard[move] = self.playerType

            # Score move
            # Note: Current player is updated with '-'
            moveValue = self.playerType * self._minmax(board = testBoard, currentPlayer = -self.playerType)

            scoreTracker[move] = moveValue

            # Update best move
            if moveValue > bestValue:
                bestMove = move
                bestValue = moveValue

        # Add some randomness to player were possible
        # If a draw is the highest score, the draw value selected is random
        if bestValue == 0:
            possMoves = np.where(scoreTracker == 0)[0]
            bestMove = np.random.choice(possMoves)
            
        # print(scoreTracker)

        return bestMove
    
    

In [110]:
pvalue = -1

player2X = MinMaxPlayer(playerType = pvalue, solutionMatrix=solMat)

testBoard = np.array([1, -1, 1, 0, 0, 0, 0, 0, 0], dtype=np.int8)

move = player2X.nextMove(testBoard)

display_board_characters(testBoard)
print(move)
testBoard[move] = pvalue
display_board_characters(testBoard)



X | O | X
  |   |  
  |   |  

4
X | O | X
  | O |  
  |   |  



In [129]:
player1X = MinMaxPlayer(playerType = 1, solutionMatrix=solMat)
# player1X = RandomPlayer()
# player2O = MinMaxPlayer(playerType = -1, solutionMatrix=solMat)
player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

Game is won by X (1), first move was 1
  | O | X
  | O | X
  |   | X

Game is won by X (1), first move was 1
X | X | X
  | O |  
  | O |  

Game is won by X (1), first move was 1
X |   |  
X | O | O
X |   |  

Game is won by X (1), first move was 1
X | X | O
O | X |  
  | O | X

Game is won by X (1), first move was 1
O |   |  
X | X | X
O | O | X

